# Cascade JEPA for NOUL questions

**Stage 1: the question reads the state.**
- **Online input:** question + state encoded **together** (frozen ModernBERT token states), read by a trainable predictor → ẑ₁.
- **Target Q·S:** the question encoded **alone** → q (one vector); the state encoded **alone** → its token vectors Sᵢ. Q·S = `Σ softmax(q·Sᵢ / √d) · Sᵢ`, the state summarised from this question's point of view.
- **Loss:** 1 − cos(ẑ₁, Q·S).

**Stage 2: the choice reads stage 1.** A small head maps ẑ₁ → ẑ₂; each choice is scored `cos(ẑ₂, t_choice) / τ`, with t_yes / t_no = ModernBERT's encoding of `yes` × 64 / `no` × 64 (word tokens only). Softmax over the choices, KL to the gold.

**Cascade:** stage 1 is trained first; then stage 2 is trained with stage 1's loss kept on (weight `stage1_weight_in_stage2`).

**Control:** the same model and stage 2 without any stage-1 loss. If the cascade beats it, predicting Q·S helps.

**Before training:** a check that Q·S carries the question. Many records ask several questions about the same state; their Q·S targets must differ, and q's attention over the state must not be flat.

Everything frozen is encoded once, so the heads train in minutes. Measures: validation, our test split, and the 600 Typed Decisions NOUL questions (Signpost-D-4B: 0.783 accuracy).

## Settings

In [ ]:
import os


def setting(name, default):
    value = os.environ.get(name.upper())
    if value is None:
        return default
    if isinstance(default, list):
        kind = type(default[0]) if default else str
        return [kind(v.strip()) for v in value.split(",") if v.strip()]
    if isinstance(default, bool):
        return value.strip().lower() in ("1", "true", "yes")
    return type(default)(value)


CONFIG = {
    "seed": setting("seed", 42),
    "model_name": setting("model_name", "answerdotai/ModernBERT-base"),
    "data_path": setting("data_path", "data/signpost_d_decisions/train.jsonl"),
    "data_url": setting("data_url", "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/"
                                    "decider-jepa/data/signpost_d_decisions/train.jsonl"),
    "max_sequence_length": setting("max_sequence_length", 1024),
    "split_seed": setting("split_seed", 42),
    "train_ratio": setting("train_ratio", 0.80),
    "validation_ratio": setting("validation_ratio", 0.10),
    "encode_batch": setting("encode_batch", 16),

    # Q·S target
    "qs_variant": setting("qs_variant", "auto"),      # "space|sharpness|residual", e.g. "standardised|16|True", or "auto"
    "qs_sharpness": setting("qs_sharpness", [1.0, 4.0, 16.0]),   # attention scores multiplied by this

    "qs_gate_same_state_cos": setting("qs_gate_same_state_cos", 0.95),   # same state, other question: below this

    # choice targets (stage 2)
    "target_words": setting("target_words", ["yes", "no"]),
    "target_repeats": setting("target_repeats", 64),
    "temperature": setting("temperature", 0.1),

    # predictor (stage 1): a small transformer over the joint token states
    "pred_width": setting("pred_width", 384),
    "pred_layers": setting("pred_layers", 2),
    "pred_heads": setting("pred_heads", 6),
    "dropout": setting("dropout", 0.1),

    # training
    "batch_size": setting("batch_size", 32),
    "lr": setting("lr", 3e-4),
    "weight_decay": setting("weight_decay", 0.01),
    "stage1_epochs": setting("stage1_epochs", 15),
    "stage2_epochs": setting("stage2_epochs", 30),
    "stage1_weight_in_stage2": setting("stage1_weight_in_stage2", 1.0),
    "arms": setting("arms", ["cascade", "control"]),

    "td_dataset": setting("td_dataset", "LocalLLaMA/typed-decisions"),
    "td_file": setting("td_file", "all/test-00000-of-00001.parquet"),
    "out_dir": setting("out_dir", "cascade"),
}

for k, v in CONFIG.items():
    print(f"{k:26s}: {v}")

## Data: NOUL questions, the group-aware split (as in `decoder_jepa`), and the Typed Decisions test

In [ ]:
import json
import random
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PATH = Path(CONFIG["data_path"])
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CONFIG["data_url"], DATA_PATH)


def parsed(value):
    return json.loads(value) if isinstance(value, str) else value


def noul_samples(records):
    out = []
    for record in records:
        state_text = "State:\n" + str(record["state"])
        for qid, q in record["questions"].items():
            if str(q.get("type", "")).lower() != "noul":
                continue
            gold = {str(k): float(v) for k, v in record["gold"][qid]["probabilities"].items()}
            question_text = "Question:\n" + str(q.get("instructions"))
            out.append({"group": record["id"], "question_id": qid,
                        "state_text": state_text, "question_text": question_text,
                        "joint_text": state_text + "\n\n" + question_text,
                        "gold": [gold["true"], gold["false"]]})
    return out


records = [json.loads(line) for line in DATA_PATH.open(encoding="utf-8") if line.strip()]
samples = noul_samples(records)

groups = pd.Series([s["group"] for s in samples]).drop_duplicates().to_numpy()
shuffled = np.random.default_rng(CONFIG["split_seed"]).permutation(groups)
n_train = int(len(groups) * CONFIG["train_ratio"])
n_val = int(len(groups) * CONFIG["validation_ratio"])
split_of = {g: "train" for g in shuffled[:n_train]}
split_of.update({g: "validation" for g in shuffled[n_train:n_train + n_val]})
split_of.update({g: "test" for g in shuffled[n_train + n_val:]})
splits = {name: [s for s in samples if split_of[s["group"]] == name] for name in ("train", "validation", "test")}

td_frame = pd.read_parquet(hf_hub_download(CONFIG["td_dataset"], CONFIG["td_file"], repo_type="dataset"))
td_records = [{"id": r["id"], "state": parsed(r["state"]), "questions": parsed(r["questions"]),
               "gold": parsed(r["gold"])} for r in td_frame.to_dict("records")]
splits["td_test"] = noul_samples(td_records)

print({k: len(v) for k, v in splits.items()})

## Frozen ModernBERT: encode once

- **Joint** question + state → token states (the online input), kept in fp16 on the CPU.
- **Question alone** → q, the mean over its word tokens.
- **State alone** → its word-token vectors Sᵢ (`[CLS]`/`[SEP]` left out), one encoding per record.
- **Choice targets:** `yes` × 64 and `no` × 64, mean over the word tokens.

In [ ]:
import math
import time

import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
encoder = AutoModel.from_pretrained(CONFIG["model_name"]).to(device).eval()
HIDDEN = encoder.config.hidden_size


@torch.no_grad()
def token_states(texts, words_only):
    """-> list of [tokens, hidden] fp16 CPU tensors (word tokens only if words_only)."""
    out = [None] * len(texts)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    for s in range(0, len(order), CONFIG["encode_batch"]):
        idx = order[s:s + CONFIG["encode_batch"]]
        batch = tokenizer([texts[i] for i in idx], padding=True, truncation=True,
                          max_length=CONFIG["max_sequence_length"], return_tensors="pt",
                          return_special_tokens_mask=True)
        special = batch.pop("special_tokens_mask").bool()
        states = encoder(**batch.to(device)).last_hidden_state.float().cpu()
        keep = batch["attention_mask"].bool().cpu()
        if words_only:
            keep = keep & ~special
        for row, i in enumerate(idx):
            out[i] = states[row][keep[row]].half()
    return out


t0 = time.perf_counter()
encoded = {}
for name, rows in splits.items():
    joint = token_states([r["joint_text"] for r in rows], words_only=False)
    question = [t.float().mean(0) for t in token_states([r["question_text"] for r in rows], words_only=True)]
    unique_states = sorted({r["state_text"] for r in rows})
    state_of = dict(zip(unique_states, token_states(unique_states, words_only=True)))
    encoded[name] = {"joint": joint, "q": torch.stack(question),
                     "S": [state_of[r["state_text"]] for r in rows],
                     "gold": torch.tensor([r["gold"] for r in rows]),
                     "group": [r["group"] for r in rows]}
    print(f"{name:10s}: {len(rows)} questions, {len(unique_states)} states, "
          f"joint tokens median {int(np.median([len(t) for t in joint]))}")

answer_vectors = [t.float().mean(0) for t in token_states(
    [" ".join([w] * CONFIG["target_repeats"]) for w in CONFIG["target_words"]], words_only=True)]
CHOICES = torch.stack(answer_vectors).to(device)          # [true, false]
c = F.cosine_similarity(CHOICES[0], CHOICES[1], dim=0).item()
print(f"choice targets {CONFIG['target_words']} x {CONFIG['target_repeats']}: cos {c:.4f}, highest gold "
      f"probability {1 / (1 + math.exp(-(1 - c) / CONFIG['temperature'])):.4f}")
print(f"encoded in {time.perf_counter() - t0:.0f} s")

del encoder
torch.cuda.empty_cache()

## The Q·S target, and the check that it carries the question

`Q·S = Σ softmax(q·Sᵢ / √d) · Sᵢ`, computed either in ModernBERT's raw space or after **standardising** every dimension with the mean and std over all training state tokens (`standardised`).

Why standardise: ModernBERT's token vectors share one large common direction, so any weighted average of them is nearly the same vector, whichever tokens the question picks (the first run of this check: same-state cos 0.994 in the raw space). Subtracting a mean from the keys alone does not help: it shifts every score by the same amount, which softmax ignores. Standardising removes the common direction from q, the keys and the values together.

The check, per key choice:
- **Same state, different question:** cosine between the Q·S of two questions on the same record. If Q·S carries the question, this is clearly below 1.
- **Different states:** the same cosine for questions on different records, for scale.
- **Attention sharpness:** entropy of q's attention over the state tokens, as a fraction of the flat maximum (1.0 = flat).

In [ ]:
all_state_tokens = torch.cat([t.float() for t in {id(t): t for t in encoded["train"]["S"]}.values()])
TOKEN_MEAN = all_state_tokens.mean(0)
TOKEN_STD = all_state_tokens.std(0).clamp_min(1e-6)
del all_state_tokens


def qs_target(q, S, variant):
    """variant = (space, sharpness, residual). residual: subtract the state's plain average, which every
    question on the same state shares, so only what the question picks remains."""
    space, sharpness, residual = variant
    S = S.float()
    if space == "standardised":
        S = (S - TOKEN_MEAN) / TOKEN_STD
        q = (q - TOKEN_MEAN) / TOKEN_STD
    weights = torch.softmax(sharpness * (S @ q) / math.sqrt(S.shape[1]), dim=0)
    entropy = -(weights * weights.clamp_min(1e-12).log()).sum() / math.log(len(weights))
    target = weights @ S
    if residual:
        target = target - S.mean(0)
    return target, entropy.item()


def qs_all(name, variant):
    e = encoded[name]
    out = [qs_target(e["q"][i], e["S"][i], variant) for i in range(len(e["S"]))]
    return torch.stack([t for t, _ in out]), np.array([h for _, h in out])


def qs_check(variant):
    targets, entropy = qs_all("train", variant)
    groups = encoded["train"]["group"]
    by_group = {}
    for i, g in enumerate(groups):
        by_group.setdefault(g, []).append(i)
    same = []
    for idx in by_group.values():
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                same.append(F.cosine_similarity(targets[idx[a]], targets[idx[b]], dim=0).item())
    rng = np.random.default_rng(0)
    pairs = rng.integers(0, len(targets), size=(3000, 2))
    other = [F.cosine_similarity(targets[a], targets[b], dim=0).item()
             for a, b in pairs if groups[a] != groups[b]]
    return {"space": variant[0], "sharpness": variant[1], "residual": variant[2],
            "same_state_cos": np.mean(same), "same_state_pairs": len(same),
            "different_state_cos": np.mean(other), "attention_entropy": entropy.mean()}


# how alike are the questions themselves (q vectors) on the same state?
q_train, groups_train = encoded["train"]["q"], encoded["train"]["group"]
q_same = [F.cosine_similarity(q_train[a], q_train[b], dim=0).item()
          for a in range(len(q_train)) for b in range(a + 1, min(a + 6, len(q_train)))
          if groups_train[a] == groups_train[b]]
print(f"q vectors, same state: mean cos {np.mean(q_same):.4f} ({len(q_same)} pairs)")

variants = [(space, sharp, residual) for space in ("raw", "standardised")
            for sharp in CONFIG["qs_sharpness"] for residual in (False, True)]
qs_report = pd.DataFrame([qs_check(v) for v in variants])
qs_report["separation"] = qs_report["different_state_cos"] - qs_report["same_state_cos"]
display(qs_report.sort_values("same_state_cos").round(4))

if CONFIG["qs_variant"] == "auto":
    best = qs_report.sort_values("same_state_cos").iloc[0]
    QS_KEYS = (best["space"], float(best["sharpness"]), bool(best["residual"]))
else:
    space, sharp, residual = CONFIG["qs_variant"].split("|")
    QS_KEYS = (space, float(sharp), residual == "True")

chosen = qs_report[(qs_report["space"] == QS_KEYS[0]) & (qs_report["sharpness"] == QS_KEYS[1])
                   & (qs_report["residual"] == QS_KEYS[2])].iloc[0]
print(f"\nQ·S variant: {QS_KEYS} | same-state cos {chosen['same_state_cos']:.4f} "
      f"| different-state cos {chosen['different_state_cos']:.4f} "
      f"(gate < {CONFIG['qs_gate_same_state_cos']}) | attention entropy {chosen['attention_entropy']:.3f}")
assert chosen["same_state_cos"] < CONFIG["qs_gate_same_state_cos"], (
    "Q·S hardly changes with the question: the stage-1 target would not carry the question."
)
print("Q·S PASSES THE CHECK")

for name in encoded:
    encoded[name]["qs"], _ = qs_all(name, QS_KEYS)

## The model

- **Stage 1 predictor:** the joint token states (768) → a linear map to `pred_width`, a learned query token prepended, `pred_layers` transformer layers, and the query token's output → linear → ẑ₁ (768).
- **Stage 2 head:** ẑ₁ → LayerNorm → Linear → GELU → Linear → ẑ₂ (768); scores `cos(ẑ₂, t_choice) / τ`.

In [ ]:
import copy

import torch.nn as nn


class CascadeModel(nn.Module):
    def __init__(self):
        super().__init__()
        w = CONFIG["pred_width"]
        self.inp = nn.Linear(HIDDEN, w)
        self.query = nn.Parameter(torch.randn(1, 1, w) * 0.02)
        layer = nn.TransformerEncoderLayer(w, CONFIG["pred_heads"], 4 * w, CONFIG["dropout"],
                                           batch_first=True, norm_first=True, activation="gelu")
        self.layers = nn.TransformerEncoder(layer, CONFIG["pred_layers"])
        self.z1 = nn.Sequential(nn.LayerNorm(w), nn.Linear(w, HIDDEN))
        self.stage2 = nn.Sequential(nn.LayerNorm(HIDDEN), nn.Linear(HIDDEN, HIDDEN), nn.GELU(),
                                    nn.Linear(HIDDEN, HIDDEN))

    def forward(self, tokens, pad_mask):
        x = self.inp(tokens)
        x = torch.cat([self.query.expand(len(x), -1, -1), x], dim=1)
        mask = torch.cat([torch.zeros(len(x), 1, dtype=torch.bool, device=x.device), pad_mask], dim=1)
        z1 = self.z1(self.layers(x, src_key_padding_mask=mask)[:, 0])
        z2 = self.stage2(z1)
        logits = F.cosine_similarity(z2.unsqueeze(1), CHOICES.unsqueeze(0), dim=-1) / CONFIG["temperature"]
        return z1, logits


def batch_of(name, idx):
    e = encoded[name]
    seqs = [e["joint"][i] for i in idx]
    length = max(len(t) for t in seqs)
    tokens = torch.zeros(len(seqs), length, HIDDEN, dtype=torch.float16)
    pad = torch.ones(len(seqs), length, dtype=torch.bool)
    for r, t in enumerate(seqs):
        tokens[r, :len(t)] = t
        pad[r, :len(t)] = False
    return (tokens.to(device).float(), pad.to(device),
            e["qs"][idx].to(device), e["gold"][idx].to(device))


def kl(logits, gold):
    return (gold * (gold.clamp_min(1e-12).log() - F.log_softmax(logits, dim=1))).sum(1)


@torch.no_grad()
def evaluate(model, name):
    model.eval()
    n = len(encoded[name]["joint"])
    order = sorted(range(n), key=lambda i: len(encoded[name]["joint"][i]))
    z_cos, logits_all, gold_all = [], [], []
    for s in range(0, n, CONFIG["batch_size"]):
        idx = order[s:s + CONFIG["batch_size"]]
        tokens, pad, qs, gold = batch_of(name, idx)
        z1, logits = model(tokens, pad)
        z_cos.append(F.cosine_similarity(z1, qs, dim=1))
        logits_all.append(logits)
        gold_all.append(gold)
    logits, gold, z_cos = torch.cat(logits_all), torch.cat(gold_all), torch.cat(z_cos)
    p = F.softmax(logits, dim=1)
    uniform = torch.full_like(gold, 0.5)
    return {"stage1_cos": z_cos.mean().item(),
            "kl": kl(logits, gold).mean().item(),
            "uniform_kl": kl(uniform.log(), gold).mean().item(),
            "top_choice": (p.argmax(1) == gold.argmax(1)).float().mean().item(),
            "brier": ((p - gold) ** 2).sum(1).mean().item(),
            "p_true_std": p[:, 0].std().item()}


# the stage-1 baseline: always predicting the training split's mean Q·S
mean_qs = encoded["train"]["qs"].mean(0)
for name in ("validation", "td_test"):
    base = F.cosine_similarity(encoded[name]["qs"], mean_qs.unsqueeze(0), dim=1).mean().item()
    print(f"{name}: cos(Q·S, mean training Q·S) = {base:.4f}  <- stage 1 must beat this")

## Training

- **cascade:** stage 1 alone for `stage1_epochs` (loss 1 − cos(ẑ₁, Q·S)), then stage 2 for `stage2_epochs` (KL + `stage1_weight_in_stage2` × stage-1 loss).
- **control:** stage 2 only (KL), the same model, no stage-1 loss.

The best stage-2 epoch by validation KL is kept.

In [ ]:
def run_epochs(model, opt, epochs, stage1_weight, kl_weight, label):
    n = len(encoded["train"]["joint"])
    best = (float("inf"), None, 0)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n).tolist()
        total = 0.0
        for s in range(0, n, CONFIG["batch_size"]):
            idx = perm[s:s + CONFIG["batch_size"]]
            tokens, pad, qs, gold = batch_of("train", idx)
            z1, logits = model(tokens, pad)
            loss = (stage1_weight * (1 - F.cosine_similarity(z1, qs, dim=1)).mean()
                    + kl_weight * kl(logits, gold).mean())
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(idx)
        v = evaluate(model, "validation")
        history.append({"arm": label, "epoch": epoch, "train_loss": total / n, **{f"val_{k}": x for k, x in v.items()}})
        print(f"{label} epoch {epoch:2d} | train {total / n:.4f} | val stage-1 cos {v['stage1_cos']:.4f} | "
              f"val KL {v['kl']:.4f} (uniform {v['uniform_kl']:.4f}) | top {v['top_choice']:.3f} | "
              f"P(true) std {v['p_true_std']:.3f}")
        if kl_weight > 0 and v["kl"] < best[0]:
            best = (v["kl"], copy.deepcopy(model.state_dict()), epoch)
    if best[1] is not None:
        model.load_state_dict(best[1])
    return history, best[2]


results, histories = {}, []

for arm in CONFIG["arms"]:
    torch.manual_seed(CONFIG["seed"])
    model = CascadeModel().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    if arm == "cascade":
        h, _ = run_epochs(model, opt, CONFIG["stage1_epochs"], 1.0, 0.0, "cascade stage 1")
        histories += h
        h, best_epoch = run_epochs(model, opt, CONFIG["stage2_epochs"], CONFIG["stage1_weight_in_stage2"], 1.0,
                                   "cascade stage 2")
    elif arm == "control":
        h, best_epoch = run_epochs(model, opt, CONFIG["stage2_epochs"], 0.0, 1.0, "control")
    else:
        raise ValueError(arm)
    histories += h
    results[arm] = {"best_stage2_epoch": best_epoch,
                    **{split: evaluate(model, split) for split in ("train", "validation", "test", "td_test")}}
    print()

## Results

In [ ]:
rows = []
for arm, r in results.items():
    for split in ("train", "validation", "test", "td_test"):
        rows.append({"arm": arm, "split": split, "best_stage2_epoch": r["best_stage2_epoch"], **r[split]})
table = pd.DataFrame(rows)
pd.set_option("display.width", 250)
display(table.round(4))

td = table[table["split"] == "td_test"].set_index("arm")
print("\nTyped Decisions NOUL (600): Signpost-D-4B accuracy 0.7833, KL 0.0952")
for arm in td.index:
    print(f"  {arm:8s}: accuracy {td.loc[arm, 'top_choice']:.4f}, KL {td.loc[arm, 'kl']:.4f} "
          f"(uniform {td.loc[arm, 'uniform_kl']:.4f})")

os.makedirs(CONFIG["out_dir"], exist_ok=True)
table.to_csv(os.path.join(CONFIG["out_dir"], "results.csv"), index=False)
pd.DataFrame(histories).to_csv(os.path.join(CONFIG["out_dir"], "history.csv"), index=False)
qs_report.to_csv(os.path.join(CONFIG["out_dir"], "qs_check.csv"), index=False)
print("\nSaved:", CONFIG["out_dir"])